## 20. Basel III IRB Regulatory Capital

In [ ]:
# ============================================================
# CELL — Basel III IRB Capital Formula
# ============================================================
from scipy.stats import norm

# Standard Basel III asset correlation formula for retail/consumer exposures
# (simplified, fixed-correlation version used for "other retail" exposure class).
# R is typically lower and roughly constant for retail vs. the PD-dependent
# correlation formula used for corporate exposures.
R = 0.04  # Basel-prescribed asset correlation for "other retail" exposures

def basel_irb_capital(pd_val, lgd_val):
    """
    Basel III IRB capital requirement per unit exposure.
    K = LGD * N[ (1-R)^-0.5 * G(PD) + (R/(1-R))^0.5 * G(0.999) ] - PD * LGD
    """
    pd_val = np.clip(pd_val, 1e-6, 1 - 1e-6)  # avoid G(0) or G(1) -> infinity
    g_pd = norm.ppf(pd_val)
    g_999 = norm.ppf(0.999)
    inner = (1 - R) ** -0.5 * g_pd + (R / (1 - R)) ** 0.5 * g_999
    k = lgd_val * norm.cdf(inner) - pd_val * lgd_val
    return np.maximum(k, 0)  # capital can't be negative

# Apply to Stage 1 loans
capital_stage1 = basel_irb_capital(
    staging_df.loc[common_idx_1, 'current_pd'].values,
    predicted_lgd_stage1.reindex(common_idx_1).values
)
capital_stage1 = pd.Series(capital_stage1, index=common_idx_1)

# Apply to Stage 2 loans
capital_stage2 = basel_irb_capital(
    lifetime_pd_stage2.reindex(common_idx_2).values,
    predicted_lgd_stage2.reindex(common_idx_2).values
)
capital_stage2 = pd.Series(capital_stage2, index=common_idx_2)

# Capital requirement is a percentage of EAD -- multiply through
capital_dollars_stage1 = capital_stage1 * ead_stage1.reindex(common_idx_1)
capital_dollars_stage2 = capital_stage2 * ead_stage2.reindex(common_idx_2)

print("--- Basel III IRB Capital Requirement ---")
print(f"Stage 1: mean capital ratio = {capital_stage1.mean():.4f}, "
      f"total capital required = ${capital_dollars_stage1.sum():,.0f}")
print(f"Stage 2: mean capital ratio = {capital_stage2.mean():.4f}, "
      f"total capital required = ${capital_dollars_stage2.sum():,.0f}")

total_capital = capital_dollars_stage1.sum() + capital_dollars_stage2.sum()
total_provisions = provisions_df['ecl'].sum()

print(f"\nTotal regulatory capital required: ${total_capital:,.0f}")
print(f"Total IFRS 9 provisions (expected loss): ${total_provisions:,.0f}")

In [ ]:
# ============================================================
# CORRECTED — Provisions vs. Capital, side by side (no incorrect subtraction)
# ============================================================
print("--- IFRS 9 Provisions vs. Basel III Capital: Side-by-Side Comparison ---")
print(f"IFRS 9 provisions (expected loss):              ${total_provisions:,.0f}")
print(f"Basel III IRB capital (unexpected loss buffer): ${total_capital:,.0f}")
print(f"Combined total loss-absorbing resources:        ${total_provisions + total_capital:,.0f}")
print()
print("These are NOT subtracted from each other -- the Basel III formula already")
print("nets out the expected-loss component internally (the '- PD x LGD' term at")
print("the end of the K formula). Capital already represents ONLY the unexpected-")
print("loss cushion, sitting on top of, not carved out of, the provisions above.")
print()
print(f"As a sanity ratio: capital is {total_capital/total_provisions:.2f}x the size of provisions")
print("for this portfolio -- i.e., the unexpected-loss cushion regulators require is")
print("roughly comparable in magnitude to the expected-loss reserve already booked.")

### Bug Note — Incorrect "Buffer" Framing in the Basel III Output

The first version of the Basel III summary printed this line:

> "The 'buffer': capital required BEYOND provisions = $1,267,358,020"

This was wrong. The value printed was simply `total_capital` restated --
not `total_capital - total_provisions` as the label implied. The underlying
capital calculation itself was correct; only the explanatory print
statement mislabeled what the number represented.

**Why this matters conceptually, not just as a typo:** the Basel III IRB
formula already nets out the expected-loss component internally, via the
`- PD x LGD` term at the end of the K formula:

K = LGD x N[(1-R)^-0.5 x G(PD) + (R/(1-R))^0.5 x G(0.999)] - PD x LGD

That subtraction means `total_capital` is *already* the unexpected-loss
buffer, by construction -- it does not need provisions subtracted from it
a second time. Doing so would double-count the expected-loss deduction
that the formula already performs.

IFRS 9 provisions and Basel III capital answer two separate questions,
computed from the same PD/LGD inputs via two different formulas:
provisions cover the loss the bank *expects*; capital is the cushion held
*in addition*, for losses *beyond* what's expected. They are meant to be
viewed side by side, as two additive cushions, not netted against each
other.

**Corrected comparison:**
- IFRS 9 provisions (expected loss): $1,447,833,735
- Basel III IRB capital (unexpected-loss buffer): $1,267,358,020
- Combined loss-absorbing resources: $2,715,191,755
- Capital is roughly 0.88x the size of provisions for this portfolio --
  the unexpected-loss cushion regulators require is comparable in
  magnitude to the expected-loss reserve already booked, not an
  additional deduction from it.